In [1]:
!pip -q install -q trl evaluate datasets==3.6.0 trl==0.12.0 bitsandbytes>=0.46.1 rouge_score

!nvidia-smi

In [2]:
from peft import PeftModel
from datasets import load_dataset
from tqdm import tqdm
import torch
import evaluate
from transformers import AutoModelForCausalLM, AutoTokenizer

In [3]:
BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
ADAPTER_PATH = "pameydorke/redred-summarizer"

SYSTEM_PROMPT = (
    "You are a helpful assistant that writes concise TL;DR summaries "
    "of Reddit posts in one or two sentences."
)

In [4]:


base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)

model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

adapter_config.json: 0.00B [00:00, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/73.9M [00:00<?, ?B/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 1536)
        (layers): ModuleList(
          (0-27): 28 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=1536, out_features=1536, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1536, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=1536, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora

In [6]:
dataset = load_dataset("webis/tldr-17", split="train[:10%]", trust_remote_code=True)
dataset = dataset.train_test_split(test_size=0.005, seed=42)
eval_ds = dataset["test"]
eval_subset = eval_ds.shuffle(seed=42).select(range(min(100, len(eval_ds))))

data/corpus-webis-tldr-17.zip:   0%|          | 0.00/3.14G [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3848330 [00:00<?, ? examples/s]

In [7]:
rouge = evaluate.load("rouge")
predictions, references = [], []

for example in tqdm(eval_subset):
    content = example["content"]   # keep raw; clean_text was not shown, apply if safe
    ref = example["summary"]

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Reddit post:\n{content}"},
    ]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=64,
            do_sample=False,
            num_beams=1,               # simpler, greedy
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    pred = tokenizer.decode(
        outputs[0][inputs.input_ids.shape[1]:],
        skip_special_tokens=True,
    ).strip()

    if pred.startswith("assistant\n"):
        print('>>>', 'assistant on pred')
        pred = pred[len("assistant\n"):]

    if "<|im_end|>" in pred:
        print('>>>', '<|im_end|> on pred')
        pred = pred.replace("<|im_end|>", "").strip()

    predictions.append(pred)
    references.append(ref)

100%|██████████| 100/100 [13:01<00:00,  7.82s/it]


In [8]:
results = rouge.compute(predictions=predictions, references=references, use_stemmer=True)
for k, v in results.items():
    print(f"{k}: {v:.4f}")

rouge1: 0.1262
rouge2: 0.0177
rougeL: 0.0877
rougeLsum: 0.0895
